# 📦 Notebook 01 — Data Loading
**Healthcare RAG-Powered Medical Q&A Assistant**  
**eyouth × DEPI | Microsoft Machine Learning Track | 2025**

---

### 🎯 Objectives
- Install and verify required libraries
- Load the PubMedQA dataset from HuggingFace
- Inspect the dataset structure, columns, and data types
- Understand the schema and content of each field
- Save raw data locally for reproducibility

### 📋 Output
- `data/raw/pubmedqa_raw.csv` — raw dataset saved locally
- Dataset shape, column info, and sample records printed

---

## 0. Install Dependencies

In [1]:
# Run this cell only once to install required packages
# Comment out after first run

# !pip install datasets pandas numpy huggingface-hub tqdm

## 1. Imports & Configuration

In [2]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
RAW_DATA_DIR = '../data/raw'
os.makedirs(RAW_DATA_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(RAW_DATA_DIR, 'pubmedqa_raw.csv')

# ── Dataset config ─────────────────────────────────────────────────────────
DATASET_NAME = 'llamafactory/PubMedQA'

print('✅ Imports successful')
print(f'📁 Raw data will be saved to: {OUTPUT_PATH}')

✅ Imports successful
📁 Raw data will be saved to: ../data/raw\pubmedqa_raw.csv


## 2. Load Dataset from HuggingFace

In [3]:
print(f'⏳ Loading dataset: {DATASET_NAME} ...')

dataset = load_dataset(DATASET_NAME)

print('\n✅ Dataset loaded successfully!')
print('\n📊 Dataset overview:')
print(dataset)

⏳ Loading dataset: llamafactory/PubMedQA ...

✅ Dataset loaded successfully!

📊 Dataset overview:
DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1000
    })
})


## 3. Inspect Available Splits

In [4]:
print('🔍 Available splits:', list(dataset.keys()))
print()

for split_name, split_data in dataset.items():
    print(f'  Split: "{split_name}" → {len(split_data):,} records')

🔍 Available splits: ['train', 'test']

  Split: "train" → 10,000 records
  Split: "test" → 1,000 records


## 4. Convert to Pandas DataFrame

In [5]:
# Identify the main split (usually 'train')
main_split = list(dataset.keys())[0]
print(f'📌 Using split: "{main_split}"')

df = pd.DataFrame(dataset[main_split])

print(f'\n✅ Converted to DataFrame')
print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

📌 Using split: "train"

✅ Converted to DataFrame
   Shape: 10,000 rows × 3 columns


## 5. Explore Schema & Columns

In [6]:
print('📋 Columns and Data Types:')
print('─' * 50)
print(df.dtypes)
print()
print('📋 Column names:')
print(df.columns.tolist())

📋 Columns and Data Types:
──────────────────────────────────────────────────
instruction    object
input          object
output         object
dtype: object

📋 Column names:
['instruction', 'input', 'output']


In [7]:
print('📐 Shape:', df.shape)
print()
print('🔢 Memory usage:')
print(df.memory_usage(deep=True).sum() / 1024**2, 'MB')

📐 Shape: (10000, 3)

🔢 Memory usage:
20.32674789428711 MB


## 6. Sample Records

In [8]:
print('👀 First 3 records:')
pd.set_option('display.max_colwidth', 200)
df.head(3)

👀 First 3 records:


,instruction,input,output
0,"Answer the question based on the following context: Although the use of alternative medicine in the United States is increasing, no published studies have documented the effectiveness of naturopat...",Question: Is naturopathy as effective as conventional therapy for treatment of menopausal symptoms?,Naturopathy appears to be an effective alternative for relief of specific menopausal symptoms compared to conventional therapy.
1,"Answer the question based on the following context: To estimate the feasibility, utility and resource implications of electronically captured routine data for health technology assessment by rando...",Question: Can randomised trials rely on existing electronic data?,"Routine data have the potential to support health technology assessment by RCTs. The cost of data collection and analysis is likely to fall, although further work is required to improve the validi..."
2,Answer the question based on the following context: To compare morbidity in two groups of patients who underwent retropubic or laparoscopic radical prostatectomy in the same period. The clinical a...,Question: Is laparoscopic radical prostatectomy better than traditional retropubic radical prostatectomy?,The results of our non-randomized study show that up to now laparoscopic radical prostatectomy does not provide significant advantages in terms of peri-operative morbidity compared with the tradit...


In [9]:
# Inspect a single full record in detail
print('🔬 Full detail of record #0:')
print('─' * 60)
for col in df.columns:
    print(f'\n[{col}]')
    print(df.iloc[0][col])

🔬 Full detail of record #0:
────────────────────────────────────────────────────────────

[instruction]
Answer the question based on the following context: Although the use of alternative medicine in the United States is increasing, no published studies have documented the effectiveness of naturopathy for treatment of menopausal symptoms compared to women receiving conventional therapy in the clinical setting. To compare naturopathic therapy with conventional medical therapy for treatment of selected menopausal symptoms. A retrospective cohort study, using abstracted data from medical charts. One natural medicine and six conventional medical clinics at Community Health Centers of King County, Washington, from November 1, 1996, through July 31, 1998. Women aged 40 years of age or more with a diagnosis of menopausal symptoms documented by a naturopathic or conventional physician. Improvement in selected menopausal symptoms. In univariate analyses, patients treated with naturopathy for me

## 7. Missing Values Check

In [10]:
print('🔎 Missing Values per Column:')
print('─' * 40)

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

print(missing_df)
print()

if missing.sum() == 0:
    print('✅ No missing values found!')
else:
    print(f'⚠️  Total missing values: {missing.sum():,}')

🔎 Missing Values per Column:
────────────────────────────────────────
             Missing Count  Missing %
instruction              0        0.0
input                    0        0.0
output                   0        0.0

✅ No missing values found!


## 8. Duplicate Check

In [11]:
n_duplicates = df.duplicated().sum()
print(f'🔎 Duplicate rows: {n_duplicates:,}')

if n_duplicates == 0:
    print('✅ No duplicates found!')
else:
    print(f'⚠️  Found {n_duplicates} duplicate rows — will handle in preprocessing notebook')

🔎 Duplicate rows: 0
✅ No duplicates found!


## 9. Basic Statistics

In [12]:
# Check text column lengths (question/answer)
text_cols = df.select_dtypes(include='object').columns.tolist()

print('📏 Text Length Statistics (character count):')
print('─' * 50)

for col in text_cols:
    try:
        lengths = df[col].astype(str).str.len()
        print(f'\n[{col}]')
        print(f'  Min:    {lengths.min():,}')
        print(f'  Max:    {lengths.max():,}')
        print(f'  Mean:   {lengths.mean():,.0f}')
        print(f'  Median: {lengths.median():,.0f}')
    except Exception:
        pass

📏 Text Length Statistics (character count):
──────────────────────────────────────────────────

[instruction]
  Min:    240
  Max:    3,982
  Mean:   1,388
  Median: 1,373

[input]
  Min:    21
  Max:    339
  Mean:   107
  Median: 104

[output]
  Min:    35
  Max:    2,114
  Mean:   287
  Median: 264


In [13]:
# Check label/answer distribution if applicable
for col in df.columns:
    n_unique = df[col].nunique()
    if n_unique <= 10:
        print(f'\n📊 Value counts for "{col}" ({n_unique} unique values):')
        print(df[col].value_counts())

## 10. Save Raw Data Locally

In [14]:
df.to_csv(OUTPUT_PATH, index=False)

print(f'✅ Raw dataset saved to: {OUTPUT_PATH}')
print(f'   Rows: {df.shape[0]:,}')
print(f'   Columns: {df.shape[1]}')
print(f'   File size: {os.path.getsize(OUTPUT_PATH) / 1024**2:.1f} MB')

✅ Raw dataset saved to: ../data/raw\pubmedqa_raw.csv
   Rows: 10,000
   Columns: 3
   File size: 17.1 MB


## ✅ Summary

| Item | Result |
|---|---|
| Dataset | llamafactory/PubMedQA |
| Records loaded | *(fill after running)* |
| Columns | *(fill after running)* |
| Missing values | *(fill after running)* |
| Duplicates | *(fill after running)* |
| Saved to | `data/raw/pubmedqa_raw.csv` |

---

### ➡️ Next Step
Open **`02_preprocessing.ipynb`** to clean, normalize, and label the dataset.